# 03 - AQI Forecasting (Supervised Regression)

This notebook trains and evaluates four regression baselines on pre-split data:
- Naive baseline (`AQI_lag1`)
- Linear Regression
- Random Forest Regressor
- XGBoost Regressor

Leakage controls:
- feature schema loaded from `outputs/models/feature_columns.json`
- no fitting on test data
- scaler already applied in preprocessing step


In [ ]:
from pathlib import Path
import json
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

from xgboost import XGBRegressor

sns.set_theme(style='whitegrid')

TRAIN_PATH = Path('data/splits/train.csv')
TEST_PATH = Path('data/splits/test.csv')
FEATURES_PATH = Path('outputs/models/feature_columns.json')

REPORTS_DIR = Path('outputs/reports')
MODELS_DIR = Path('outputs/models')
FIG_DIR = Path('outputs/figures')
for d in [REPORTS_DIR, MODELS_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)


## Load Data And Feature Schema

In [ ]:
if not TRAIN_PATH.exists() or not TEST_PATH.exists():
    raise FileNotFoundError('Missing train/test split CSV files. Run 02_preprocessing first.')
if not FEATURES_PATH.exists():
    raise FileNotFoundError('Missing feature_columns.json. Run 02_preprocessing first.')

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

feature_columns = json.loads(FEATURES_PATH.read_text(encoding='utf-8'))
target_col = 'AQI'

missing_train = [c for c in feature_columns + [target_col] if c not in train_df.columns]
missing_test = [c for c in feature_columns + [target_col, 'City', 'Timestamp'] if c not in test_df.columns]
if missing_train:
    raise ValueError(f'Missing columns in train: {missing_train}')
if missing_test:
    raise ValueError(f'Missing columns in test: {missing_test}')

X_train = train_df[feature_columns].astype(float)
y_train = train_df[target_col].astype(float)
X_test = test_df[feature_columns].astype(float)
y_test = test_df[target_col].astype(float)

assert X_train.isna().sum().sum() == 0, 'NaNs found in X_train.'
assert X_test.isna().sum().sum() == 0, 'NaNs found in X_test.'
assert y_train.isna().sum() == 0 and y_test.isna().sum() == 0, 'NaNs found in target.'

test_meta = test_df[['Timestamp', 'City']].copy()
test_meta['Timestamp'] = pd.to_datetime(test_meta['Timestamp'], errors='coerce')
assert test_meta['Timestamp'].isna().sum() == 0, 'Invalid Timestamp values in test split.'

print('Train shape:', X_train.shape, '| Test shape:', X_test.shape)
print('Features loaded:', len(feature_columns))


## Metric Functions

- MAE: average absolute error in AQI points
- RMSE: square root of mean squared error, penalizes larger misses
- MAPE: mean absolute percentage error (`* 100`)


In [ ]:
def compute_metrics(y_true, y_pred):
    y_true_arr = np.asarray(y_true, dtype=float)
    y_pred_arr = np.asarray(y_pred, dtype=float)
    mae = mean_absolute_error(y_true_arr, y_pred_arr)
    rmse = np.sqrt(mean_squared_error(y_true_arr, y_pred_arr))
    mape = np.mean(np.abs((y_true_arr - y_pred_arr) / y_true_arr)) * 100.0
    return {'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

results = []
predictions = {}


## Model 1 - Naive Baseline

Practical meaning: this baseline assumes AQI tomorrow is approximately yesterday's AQI (`AQI_lag1`). Any learned model should beat this.

In [ ]:
if 'AQI_lag1' not in X_test.columns:
    raise ValueError("Feature 'AQI_lag1' missing; cannot compute naive baseline.")

y_pred_naive = X_test['AQI_lag1'].values
metrics_naive = compute_metrics(y_test, y_pred_naive)
results.append({'Model': 'Naive Baseline', **metrics_naive})
predictions['Naive Baseline'] = y_pred_naive
metrics_naive


## Model 2 - Linear Regression

Practical meaning: a linear benchmark checks whether simple additive relationships already explain AQI trends.

In [ ]:
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)
y_pred_lr = lin_reg.predict(X_test)

metrics_lr = compute_metrics(y_test, y_pred_lr)
results.append({'Model': 'Linear Regression', **metrics_lr})
predictions['Linear Regression'] = y_pred_lr
metrics_lr


## Model 3 - Random Forest Regressor

Practical meaning: a tree ensemble captures nonlinear effects and feature interactions common in pollution series.

In [ ]:
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1,
)
try:
    rf.fit(X_train, y_train)
except PermissionError:
    rf = RandomForestRegressor(
        n_estimators=200,
        max_depth=15,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=1,
    )
    rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

metrics_rf = compute_metrics(y_test, y_pred_rf)
results.append({'Model': 'Random Forest', **metrics_rf})
predictions['Random Forest'] = y_pred_rf
metrics_rf


## Model 4 - XGBoost Regressor

Practical meaning: boosted trees iteratively reduce residual error and often perform best for structured tabular forecasting.

In [ ]:
xgb = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective='reg:squarederror',
    n_jobs=-1,
)
try:
    xgb.fit(X_train, y_train)
except PermissionError:
    xgb = XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        objective='reg:squarederror',
        n_jobs=1,
    )
    xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)

metrics_xgb = compute_metrics(y_test, y_pred_xgb)
results.append({'Model': 'XGBoost', **metrics_xgb})
predictions['XGBoost'] = y_pred_xgb
metrics_xgb


## Model Comparison And Best Model Selection

Best model rule (explicit): sort by `MAE`, then `RMSE`, then `MAPE` ascending.

In [ ]:
comparison_df = pd.DataFrame(results)
comparison_df = comparison_df.sort_values(['MAE', 'RMSE', 'MAPE'], ascending=[True, True, True]).reset_index(drop=True)

best_row = comparison_df.iloc[0]
best_model_name = best_row['Model']

print('Best model:', best_model_name)
display(comparison_df)

best_per_metric = {
    'MAE': comparison_df.loc[comparison_df['MAE'].idxmin(), 'Model'],
    'RMSE': comparison_df.loc[comparison_df['RMSE'].idxmin(), 'Model'],
    'MAPE': comparison_df.loc[comparison_df['MAPE'].idxmin(), 'Model'],
}
print('Best per metric:', best_per_metric)


In [ ]:
comparison_df.to_csv(REPORTS_DIR / 'model_comparison.csv', index=False)

model_objects = {
    'Linear Regression': lin_reg,
    'Random Forest': rf,
    'XGBoost': xgb,
}

if best_model_name == 'Naive Baseline':
    joblib.dump({'type': 'naive_baseline', 'feature': 'AQI_lag1'}, MODELS_DIR / 'best_model.pkl')
else:
    joblib.dump(model_objects[best_model_name], MODELS_DIR / 'best_model.pkl')

(MODELS_DIR / 'best_model_name.txt').write_text(str(best_model_name), encoding='utf-8')

print(f"Saved: {REPORTS_DIR / 'model_comparison.csv'}")
print(f"Saved: {MODELS_DIR / 'best_model.pkl'}")
print(f"Saved: {MODELS_DIR / 'best_model_name.txt'}")


## Build Best-Model Error Frame For Plots

In [ ]:
y_pred_best = predictions[best_model_name]

eval_df = test_meta.copy()
eval_df['actual'] = y_test.values
eval_df['predicted'] = y_pred_best
eval_df['residual'] = eval_df['actual'] - eval_df['predicted']
eval_df['abs_error'] = eval_df['residual'].abs()
eval_df['month_num'] = eval_df['Timestamp'].dt.month

assert eval_df[['actual', 'predicted', 'residual', 'abs_error']].isna().sum().sum() == 0
eval_df.head()


## Plot 1 - Predicted vs Actual (First 180 Days, Date-Aggregated)

For readability across multi-city test data, values are aggregated by date (mean AQI across cities), then first 180 unique dates are plotted.

In [ ]:
daily_eval = (
    eval_df.groupby('Timestamp', as_index=False)[['actual', 'predicted']]
    .mean()
    .sort_values('Timestamp')
)
plot_180 = daily_eval.head(180)

plt.figure(figsize=(14, 6))
plt.plot(plot_180['Timestamp'], plot_180['actual'], label='Actual AQI', linewidth=2)
plt.plot(plot_180['Timestamp'], plot_180['predicted'], label='Predicted AQI', linewidth=2)
plt.title(f'Predicted vs Actual AQI ({best_model_name}, first 180 days)')
plt.xlabel('Date')
plt.ylabel('AQI')
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'predicted_vs_actual.png', dpi=200)
plt.close()


## Plot 2 - Residual Distribution

In [ ]:
plt.figure(figsize=(12, 6))
sns.histplot(eval_df['residual'], bins=60, kde=True)
plt.title(f'Residual Distribution ({best_model_name})')
plt.xlabel('Residual (Actual - Predicted)')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig(FIG_DIR / 'residuals_distribution.png', dpi=200)
plt.close()


## Plot 3 - Absolute Error By Calendar Month

Month is derived from raw test `Timestamp`, not from scaled feature columns.

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(data=eval_df, x='month_num', y='abs_error', color='steelblue')
plt.title(f'Absolute Error By Month ({best_model_name})')
plt.xlabel('Month')
plt.ylabel('Absolute Error')
plt.tight_layout()
plt.savefig(FIG_DIR / 'error_by_month.png', dpi=200)
plt.close()


## Plot 4 - Absolute Error By City

In [ ]:
city_order = (
    eval_df.groupby('City', as_index=False)['abs_error']
    .median()
    .sort_values('abs_error', ascending=False)['City']
)

plt.figure(figsize=(14, 6))
sns.boxplot(data=eval_df, x='City', y='abs_error', order=city_order)
plt.title(f'Absolute Error By City ({best_model_name})')
plt.xlabel('City')
plt.ylabel('Absolute Error')
plt.xticks(rotation=25)
plt.tight_layout()
plt.savefig(FIG_DIR / 'error_by_city.png', dpi=200)
plt.close()
